# 06. Evaluation and Error Analysis

Evaluate the innovation classifier against expert-reviewed labels.

This notebook will:

1. Loads saved model classifications and human validation labels.
2. Measures agreement across the four innovation classes.
3. Computes precision, recall, and F1 scores.
4. Evaluates binary innovation-signal detection.
5. Summarizes disagreement patterns for error analysis.

In [1]:
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import ipywidgets as w
from IPython.display import display

ROOT = Path("..").resolve()
OUTPUT_DIR    = ROOT / "outputs"
EVALUATION_DIR = ROOT / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED          = 42
POSSIBLE_SAMPLE_SIZE = 18

VALID_HUMAN_LABELS = {
    "innovation",
    "possible_innovation",
    "not_innovation",
    "insufficient_evidence",
}

Classification records: 65
Successful classifications: 61
Preserved errors: 4


Load classification results

In [2]:
results_df = pd.read_csv(OUTPUT_DIR / "innovation_classification_results.csv")

print("Classification records :", len(results_df))
print("Successful             :", (results_df["innovation_presence"] != "error").sum())
print("Preserved errors       :", (results_df["innovation_presence"] == "error").sum())

valid_results_df = results_df[
    results_df["innovation_presence"] != "error"
].copy()

,classification,count,percent
0,possible_innovation,44,72.1
1,not_innovation,9,14.8
2,innovation,5,8.2
3,insufficient_evidence,3,4.9


Classification summary

classification_summary = (
    valid_results_df["innovation_presence"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="count")
)
classification_summary["percent"] = (
    classification_summary["count"] / len(valid_results_df) * 100
).round(1)

classification_summary

Top 15 by confidence

In [ ]:
valid_results_df[
    ["projectid", "type", "number", "title",
     "innovation_presence", "innovation_types",
     "novelty_basis", "maturity_stage", "confidence"]
].sort_values("confidence", ascending=False).head(15)

## 3. Create the expert-validation sample

A stratified sample is created for manual expert review.

Rare predicted classes are retained in full, while the larger
`possible_innovation` class is sampled reproducibly.

The validation sample is used to assess classification accuracy and identify
false positives, false negatives, and ambiguous cases.

In [4]:
validation_parts = []

for label in ["innovation", "not_innovation", "insufficient_evidence"]:
    validation_parts.append(
        valid_results_df[valid_results_df["innovation_presence"] == label]
    )

possible_pool = valid_results_df[
    valid_results_df["innovation_presence"] == "possible_innovation"
]
validation_parts.append(
    possible_pool.sample(
        n=min(POSSIBLE_SAMPLE_SIZE, len(possible_pool)),
        random_state=RANDOM_SEED,
    )
)

validation_df = (
    pd.concat(validation_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

print("Validation sample size:", len(validation_df))
print()
print(validation_df["innovation_presence"].value_counts())

Validation sample size: 35

innovation_presence
possible_innovation      18
not_innovation            9
innovation                5
insufficient_evidence     3
Name: count, dtype: int64


Attach source text

In [ ]:
units_df = pd.read_csv(
    ROOT / "data" / "processed" / "pad_components_subcomponents.csv"
)

def normalize_unit_number(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.endswith(".0"):
        value = value[:-2]
    return value

validation_df["number_key"] = validation_df["number"].apply(normalize_unit_number)
units_df["number_key"]      = units_df["number"].apply(normalize_unit_number)

validation_df = validation_df.merge(
    units_df[["projectid", "type", "number_key", "text"]],
    on=["projectid", "type", "number_key"],
    how="left",
    validate="many_to_one",
)

assert validation_df["text"].notna().all(), "Some records missing source text!"
print("Records with source text:", validation_df["text"].notna().sum())

Initialise annotation columns

In [ ]:
# --------------------------------------------------
# ADD EXPERT-ANNOTATION FIELDS
# --------------------------------------------------
validation_df["human_innovation_presence"] = ""
validation_df["human_innovation_types"]    = ""
validation_df["human_novelty_basis"]       = ""
validation_df["human_maturity_stage"]      = ""
validation_df["human_notes"]               = ""

print("Annotation columns ready.")

Interactive labeller (run once)

In [57]:
# --------------------------------------------------
# INTERACTIVE LABELLER  (run once)
# --------------------------------------------------

import ipywidgets as w
from IPython.display import display

VALID_LABELS = [
    "",                    # blank = not yet labelled
    "innovation",
    "possible_innovation",
    "not_innovation",
    "insufficient_evidence",
]

def make_labeller(df):
    state = {"i": 0}
    n = len(df)

    # --- widgets ---
    progress   = w.HTML()
    meta       = w.HTML()
    model_box  = w.HTML()
    source_txt = w.Textarea(
        disabled=True,
        layout=w.Layout(width="100%", height="220px")
    )
    label_dd   = w.Dropdown(options=VALID_LABELS, description="Label:")
    notes_ta   = w.Textarea(
        placeholder="Notes (optional)",
        layout=w.Layout(width="100%", height="70px")
    )
    status_msg = w.HTML()

    prev_btn   = w.Button(description="◀ Prev")
    next_btn   = w.Button(description="Next ▶")
    save_btn   = w.Button(description="Save & Next ▶", button_style="success")

    # --- load a record into the panel ---
    def load(i):
        row = df.iloc[i]
        current = row["human_innovation_presence"] or ""

        progress.value = (
            f"<b>Record {i+1} of {n}</b> &nbsp;|&nbsp; "
            f"Labelled: {(df['human_innovation_presence'] != '').sum()} / {n}"
        )
        meta.value = (
            f"<b>Project:</b> {row['projectid']} &nbsp; "
            f"<b>Unit:</b> {row['type']} {row['number']} &nbsp; "
            f"<b>Title:</b> {row['title']}"
        )
        model_box.value = (
            f"<b>Model presence:</b> {row['innovation_presence']} &nbsp; "
            f"<b>Confidence:</b> {row['confidence']}<br>"
            f"<b>Types:</b> {row['innovation_types']}<br>"
            f"<b>Novelty:</b> {row['novelty_basis']}"
        )
        source_txt.value = str(row.get("text", ""))
        label_dd.value   = current
        notes_ta.value   = row.get("human_notes", "") or ""

        if current:
            status_msg.value = f'<span style="color:green">✔ {current}</span>'
        else:
            status_msg.value = '<span style="color:#e07000">not yet labelled</span>'

    def save_and_next(_):
        i = state["i"]
        if not label_dd.value:
            status_msg.value = '<span style="color:red">Pick a label first.</span>'
            return
        df.loc[i, "human_innovation_presence"] = label_dd.value
        df.loc[i, "human_notes"]               = notes_ta.value
        status_msg.value = f'<span style="color:green">✔ Saved: {label_dd.value}</span>'
        if i + 1 < n:
            state["i"] = i + 1
            load(state["i"])

    def go_prev(_):
        if state["i"] > 0:
            state["i"] -= 1
            load(state["i"])

    def go_next(_):
        if state["i"] < n - 1:
            state["i"] += 1
            load(state["i"])

    save_btn.on_click(save_and_next)
    prev_btn.on_click(go_prev)
    next_btn.on_click(go_next)

    load(0)

    display(w.VBox([
        progress,
        meta,
        model_box,
        source_txt,
        w.HBox([label_dd, status_msg]),
        notes_ta,
        w.HBox([prev_btn, next_btn, save_btn]),
    ]))

make_labeller(validation_df)

In [ ]:
# --------------------------------------------------
# SAVE LABELLED VALIDATION RESULTS
# --------------------------------------------------

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = EVALUATION_DIR / f"validation_labelled_{timestamp}.csv"

validation_df.to_csv(save_path, index=False)

print(f"Saved to: {save_path}")
print(f"Total records : {len(validation_df)}")
print(f"Labelled      : {(validation_df['human_innovation_presence'] != '').sum()}")
print(f"Unlabelled    : {(validation_df['human_innovation_presence'] == '').sum()}")

## 5. Evaluate model predictions against expert labels

The manually reviewed validation sample is now compared with the model predictions.

Evaluation is performed only on records with completed human labels.

In [75]:
# --------------------------------------------------
# EVALUATION METRICS
# --------------------------------------------------

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

POSITIVE_LABELS = {"innovation", "possible_innovation"}

LABEL_ORDER = [
    "innovation",
    "possible_innovation",
    "not_innovation",
    "insufficient_evidence",
]

# Filter to labelled records only
labelled_mask = validation_df["human_innovation_presence"].fillna("").ne("")
eval_df = validation_df[labelled_mask].copy()

print(f"Total records : {len(validation_df)}")
print(f"Labelled      : {labelled_mask.sum()}")
print(f"Unlabelled    : {(~labelled_mask).sum()}")
print(f"Used for eval : {len(eval_df)}")
print()

# Binary signals
eval_df["human_binary"] = eval_df["human_innovation_presence"].isin(POSITIVE_LABELS)
eval_df["model_binary"] = eval_df["innovation_presence"].isin(POSITIVE_LABELS)

# Overall
overall_accuracy   = (eval_df["human_innovation_presence"] == eval_df["innovation_presence"]).mean()
report     = classification_report(
    eval_df["human_innovation_presence"],
    eval_df["innovation_presence"],
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)
macro_f1    = report["macro avg"]["f1-score"]
weighted_f1 = report["weighted avg"]["f1-score"]

print(f"Exact agreement : {accuracy:.1%}")
print(f"Macro F1        : {macro_f1:.3f}")
print(f"Weighted F1     : {weighted_f1:.3f}")
print()

# Binary
binary_precision = precision_score(eval_df["human_binary"], eval_df["model_binary"])
binary_recall    = recall_score(eval_df["human_binary"],    eval_df["model_binary"])
binary_f1        = f1_score(eval_df["human_binary"],        eval_df["model_binary"])

print(f"Binary precision : {binary_precision:.3f}")
print(f"Binary recall    : {binary_recall:.3f}")
print(f"Binary F1        : {binary_f1:.3f}")
print()

# Per-class metrics table
metrics_df = (
    pd.DataFrame(report).transpose()
    .reset_index()
    .rename(columns={"index": "class"})
)
display(metrics_df[["class", "precision", "recall", "f1-score", "support"]])
print()

# Confusion matrix
display(pd.crosstab(
    eval_df["human_innovation_presence"],
    eval_df["innovation_presence"],
    rownames=["Human label"],
    colnames=["Model prediction"],
    margins=True,
))

Total records : 35
Labelled      : 35
Unlabelled    : 0
Used for eval : 35

Exact agreement : 74.3%
Macro F1        : 0.785
Weighted F1     : 0.725

Binary precision : 1.000
Binary recall    : 0.885
Binary F1        : 0.939



,class,precision,recall,f1-score,support
0,innovation,1.000000,0.416667,0.588235,12.000000
1,possible_innovation,0.666667,0.857143,0.750000,14.000000
2,not_innovation,0.666667,1.000000,0.800000,6.000000
3,insufficient_evidence,1.000000,1.000000,1.000000,3.000000
4,accuracy,0.742857,0.742857,0.742857,0.742857
5,macro avg,0.833333,0.818452,0.784559,35.000000
6,weighted avg,0.809524,0.742857,0.724538,35.000000


Model prediction,innovation,insufficient_evidence,not_innovation,possible_innovation,All
Human label,,,,,
innovation,5,0,1,6,12
insufficient_evidence,0,3,0,0,3
not_innovation,0,0,6,0,6
possible_innovation,0,0,2,12,14
All,5,3,9,18,35


## 6. Classification metrics

Precision, recall, and F1 are calculated for each innovation-presence class.

These metrics show where the classifier performs well and where it tends to
over- or under-classify innovation signals.

In [76]:
# --------------------------------------------------
# ERROR ANALYSIS
# --------------------------------------------------

def classify_error(row):
    human = row["human_innovation_presence"]
    model = row["innovation_presence"]

    if human == "innovation" and model == "possible_innovation":
        return "underclassified_innovation"
    if human in POSITIVE_LABELS and model not in POSITIVE_LABELS:
        return "missed_innovation_signal"
    if human not in POSITIVE_LABELS and model in POSITIVE_LABELS:
        return "false_positive_innovation"
    return "other_disagreement"

errors_df = eval_df[
    eval_df["human_innovation_presence"] != eval_df["innovation_presence"]
].copy()

errors_df["error_type"] = errors_df.apply(classify_error, axis=1)

print(f"Disagreements: {len(errors_df)}")
print()
print(errors_df["error_type"].value_counts().to_string())
print()

display(
    errors_df[[
        "projectid", "title",
        "human_innovation_presence", "innovation_presence",
        "error_type", "confidence", "human_notes", "rationale",
    ]].sort_values("error_type")
)

Disagreements: 9

error_type
underclassified_innovation    6
missed_innovation_signal      3



,projectid,title,human_innovation_presence,innovation_presence,error_type,confidence,human_notes,rationale
7,P178043,Dissemination and communication (US$8.9 million),possible_innovation,not_innovation,missed_innovation_signal,0.86,"The passage introduces open-data practices, da...",The described activities are routine dissemina...
10,P175295,Contingent Emergency Response Component (US$0 ...,possible_innovation,not_innovation,missed_innovation_signal,0.85,The project combines electricity and broadband...,The contingent emergency response component fo...
16,P151469,Improving the enabling environment (US$4.52m),innovation,not_innovation,missed_innovation_signal,0.87,The passage introduces and rolls out improved ...,The activities described (value‑chain organiza...
2,P173240,Contingent Emergency Response Component (US$0....,innovation,possible_innovation,underclassified_innovation,0.75,The passage describes the deliberate integrati...,The CERC introduces a financing/process mechan...
11,P164665,Improving Efficiency in Service Delivery (US$9...,innovation,possible_innovation,underclassified_innovation,0.71,The passage includes several concrete innovati...,While many activities are routine capacity‑bui...
14,P173813,"Case Detection, Confirmation, Contact Tracing,...",innovation,possible_innovation,underclassified_innovation,0.62,The passage describes concrete technology-enab...,The project mentions creating and deploying ne...
15,P167543,Creating a digital data platform to enable acc...,innovation,possible_innovation,underclassified_innovation,0.78,The passage introduces a digital data platform...,The project proposes a novel digital platform ...
26,P173240,Financing demand-side interventions (US$13.0 m...,innovation,possible_innovation,underclassified_innovation,0.78,"The passage introduces targeted, gender-inclus...",The subcomponent proposes device‑financing vou...
31,P180059,Digitalization of Public Services in Key Secto...,innovation,possible_innovation,underclassified_innovation,0.68,The passage describes several potentially nove...,The project describes activities that go beyon...


In [ ]:
# --------------------------------------------------
# SAVE EVALUATION OUTPUTS
# --------------------------------------------------

eval_df.to_csv(EVALUATION_DIR / "validation_evaluated.csv", index=False)
metrics_df.to_csv(EVALUATION_DIR / "classification_metrics.csv", index=False)
errors_df.to_csv(EVALUATION_DIR / "classification_errors.csv", index=False)

summary_metrics = pd.DataFrame({
    "metric": [
        "multiclass_accuracy",
        "macro_f1",
        "weighted_f1",
        "binary_precision",
        "binary_recall",
        "binary_f1",
        "validation_sample_size",
        "exact_disagreements",
    ],
    "value": [
        overall_accuracy,
        macro_f1,
        weighted_f1,
        binary_precision,
        binary_recall,
        binary_f1,
        len(eval_df),
        len(errors_df),
    ],
})

summary_metrics.to_csv(EVALUATION_DIR / "evaluation_summary.csv", index=False)

print("Saved to:", EVALUATION_DIR)
print("  validation_evaluated.csv")
print("  classification_metrics.csv")
print("  classification_errors.csv")
print("  evaluation_summary.csv")
print()
display(summary_metrics)